In [3]:
MODEL_PATH = "/work/models/best.pt"

In [4]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

# In thông tin tổng quan
print(model)

# In chi tiết kiến trúc
model.info()

# Số class
print("Number of classes:", model.model.nc)

# Tên class
print("Class names:", model.model.names)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_s

In [5]:
print("Model stride:", model.model.stride)

Model stride: tensor([ 8., 16., 32.])


In [7]:
results = model.predict("/work/models/000001.jpg")

r = results[0]

print("Boxes shape:", r.boxes.xyxy.shape)
print("Conf:", r.boxes.conf)
print("Class:", r.boxes.cls)


image 1/1 /work/models/000001.jpg: 384x640 11 persons, 6 heads, 40.6ms
Speed: 1.2ms preprocess, 40.6ms inference, 14.4ms postprocess per image at shape (1, 3, 384, 640)
Boxes shape: torch.Size([17, 4])
Conf: tensor([0.9103, 0.8571, 0.8448, 0.8145, 0.7953, 0.7688, 0.7677, 0.6864, 0.5662, 0.4886, 0.4834, 0.4819, 0.4089, 0.3919, 0.3915, 0.3491, 0.2654], device='cuda:0')
Class: tensor([0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 1.], device='cuda:0')


In [8]:
print(model.model.names)

{0: 'person', 1: 'head'}


In [11]:
from ultralytics import YOLO

model = YOLO("/work/models/best.pt")
model.export(format="onnx", opset=12, nms=True)

Ultralytics 8.4.18 🚀 Python-3.10.12 torch-2.10.0+cu128 CPU (13th Gen Intel Core i9-13900)


Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/work/models/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (85.4 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 1.3s, saved as '/work/models/best.onnx' (42.7 MB)

Export complete (1.4s)
Results saved to /work/models
Predict:         yolo predict task=detect model=/work/models/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/work/models/best.onnx imgsz=640 data=crowdhuman.yaml  
Visualize:       https://netron.app


'/work/models/best.onnx'

In [13]:
import onnx

model = onnx.load("/work/models/best.onnx")

print("=== OUTPUT INFO ===")
for output in model.graph.output:
    print("Name:", output.name)
    print("Shape:", [
        dim.dim_value for dim in output.type.tensor_type.shape.dim
    ])
    print("Type:", output.type.tensor_type.elem_type)
    print("--------------------")

=== OUTPUT INFO ===
Name: output0
Shape: [1, 300, 6]
Type: 1
--------------------


In [14]:
import cv2
import numpy as np
import onnxruntime as ort

# Load model
session = ort.InferenceSession("/work/models/best.onnx")

input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape

# Load ảnh thật
img = cv2.imread("/work/models/000001.jpg")
img = cv2.resize(img, (640, 640))
img = img[:, :, ::-1]  # BGR → RGB
img = img.astype(np.float32) / 255.0
img = np.transpose(img, (2, 0, 1))  # HWC → CHW
img = np.expand_dims(img, 0)

# Inference
outputs = session.run(None, {input_name: img})

print("Output shape:", outputs[0].shape)
print(outputs[0][:5])

Output shape: (1, 300, 6)
[[[     565.24      232.26      628.24      434.29     0.90722           0]
  [  -0.090958      134.11      44.051      539.99     0.79107           0]
  [     76.697      268.71      127.26      421.18     0.74038           0]
  ...
  [          0           0           0           0           0           0]
  [          0           0           0           0           0           0]
  [          0           0           0           0           0           0]]]


# Thông tin tiền xử lý

In [15]:
from ultralytics import YOLO
import torch

# Load model
model = YOLO("best.pt")

# Lấy số lớp (classes)
print("\n=== Number of classes ===")
print(model.model.nc)

# Lấy tên lớp
print("\n=== Class names ===")
print(model.model.names)

# Kiểm tra kích thước input mặc định
print("\n=== Model input size (stride based) ===")
print(model.model.stride)

# Kiểm tra first layer để biết số channel input
first_layer = list(model.model.model.children())[0]
print("\n=== First layer ===")
print(first_layer)

# Dummy forward để xem shape output
dummy = torch.zeros(1, 3, 640, 640)
output = model.model(dummy)
print("\n=== Output shape ===")
print(output[0].shape)


=== Number of classes ===
2

=== Class names ===
{0: 'person', 1: 'head'}

=== Model input size (stride based) ===
tensor([ 8., 16., 32.])

=== First layer ===
Conv(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)

=== Output shape ===
torch.Size([1, 6, 8400])


In [16]:
from ultralytics import YOLO

model = YOLO("/work/models/best.pt")

print("===== MODEL ARGS =====")
for k, v in model.model.args.items():
    print(f"{k}: {v}")

===== MODEL ARGS =====
task: detect
data: crowdhuman.yaml
imgsz: 640
single_cls: False
model: /work/models/best.pt


In [17]:
import torch
from ultralytics import YOLO

model = YOLO("/work/models/best.pt")

dummy = torch.ones(1, 3, 640, 640) * 255

out1 = model.model(dummy)
out2 = model.model(dummy / 255)

print("Type out1:", type(out1))
print("Length out1:", len(out1))

# YOLOv8 thường trả tuple: (pred, aux)
pred1 = out1[0] if isinstance(out1, (list, tuple)) else out1
pred2 = out2[0] if isinstance(out2, (list, tuple)) else out2

print("Pred1 shape:", pred1.shape)
print("Pred2 shape:", pred2.shape)

print("Max out1:", pred1.abs().max())
print("Max out2:", pred2.abs().max())

first_layer = model.model.model[0]
print(first_layer)

Type out1: <class 'tuple'>
Length out1: 2
Pred1 shape: torch.Size([1, 6, 8400])
Pred2 shape: torch.Size([1, 6, 8400])
Max out1: tensor(636.0225)
Max out2: tensor(637.3620)
Conv(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)


In [18]:
import torch

first_layer = model.model.model[0]

def hook_fn(module, input, output):
    print("Layer1 max:", output.abs().max().item())

hook = first_layer.register_forward_hook(hook_fn)

dummy1 = torch.ones(1, 3, 640, 640) * 255
dummy2 = torch.ones(1, 3, 640, 640)

print("---- Input 255 ----")
model.model(dummy1)

print("---- Input 1 ----")
model.model(dummy2)

hook.remove()

---- Input 255 ----
Layer1 max: 11658.4248046875
---- Input 1 ----
Layer1 max: 47.37885665893555


In [1]:
import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)
runtime = trt.Runtime(logger)

with open("/work/models/best.onnx_b1_gpu0_fp16.engine", "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())

if engine:
    print(f"Engine has {engine.num_layers} layers")
    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        shape = engine.get_tensor_shape(name)
        print(f"  Tensor '{name}': {shape}")
else:
    print("Failed to load engine!")

ModuleNotFoundError: No module named 'tensorrt'